# VLM router benchmark, round 2

Round 1: Qwen2.5-VL-3B took ~90 ms per call on an L4 but always answered "cruise" (40%, chance level).
This round asks **why**, and whether cheap fixes help:

| Method | What the model does |
|---|---|
| `direct` | image → one letter A/B/C (round-1 setup) |
| `fewshot` | same, with 5 labelled example images in the fixed prefix (cached after the first call) |
| `perceive` | two yes/no questions: *car ahead in RIGHT lane?* and *other car in LEFT lane?* → a fixed rule picks the skill |

Each method runs on two views rendered here in Colab with MetaDrive: **top-down** (as in round 1) and the **front RGB camera**
(the real target), with 50 test frames each. Models: Qwen2.5-VL-3B and 7B, plus optionally Qwen3-VL-8B.

**Runtime → Change runtime type → L4 GPU** (A100 is fine too). Then **Runtime → Run all**. Nothing to upload.
Takes roughly 25–40 min, mostly model downloads.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y libgl1 libegl1 libgl1-mesa-dri libglx-mesa0 libegl-mesa0 xvfb > /dev/null
!pip -q install vllm uv

## 1. Helper scripts (written to /content)

In [ ]:
%%writefile /content/gen_frames2.py
"""Render labelled router-decision frames from MetaDrive, in two views.

usage: python gen_frames2.py --view topdown|camera --out DIR --n N --seed S

Road: two lanes our direction. lane y=3.5 is LEFT, y=0 is RIGHT. Other cars are parked.
Classes and ground truth (label = correct router answer):
  clear      ego right lane, nothing ahead                         -> A
  overtake   ego right lane, parked car ahead right, left free      -> B
  blocked    ego right lane, car ahead right AND car in left lane   -> A (wait)
  return     ego left lane, right lane ahead clear (car passed)     -> C
  keep_left  ego left lane, car still ahead in right lane           -> B
Perception truth per frame: right_ahead (car ahead in RIGHT lane), left_car (another car in LEFT lane).
Camera only sees ahead, so in the camera view the 'blocked' left-lane car is placed ahead (12-35 m), not beside.
"""
import argparse
import json
import os

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
if os.environ.get("P3D_DISPLAY"):  # force a panda3d display backend, e.g. p3headlessgl (EGL) or pandagl (X11)
    from panda3d.core import loadPrcFileData
    loadPrcFileData("", "load-display " + os.environ["P3D_DISPLAY"])
import numpy as np
from PIL import Image
from metadrive.component.pgblock.first_block import FirstPGBlock
from metadrive.component.vehicle.vehicle_type import DefaultVehicle
from metadrive.envs.metadrive_env import MetaDriveEnv

ap = argparse.ArgumentParser()
ap.add_argument("--view", choices=["topdown", "camera"], required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--n", type=int, default=10)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--flip", type=int, default=1, help="camera only: 1 = convert BGR->RGB")
a = ap.parse_args()
os.makedirs(a.out, exist_ok=True)
rng = np.random.default_rng(a.seed)

L_LEFT, L_RIGHT, EGO_X = 3.5, 0.0, 60.0
cfg = dict(use_render=False, traffic_density=0.0, num_scenarios=1, start_seed=0, log_level=50,
           map_config=dict(type="block_sequence", config="SSSS", lane_num=2), random_spawn_lane_index=False,
           vehicle_config=dict(spawn_lane_index=(FirstPGBlock.NODE_2, FirstPGBlock.NODE_3, 1)))
if a.view == "camera":
    from metadrive.component.sensors.rgb_camera import RGBCamera
    cfg.update(image_observation=True, norm_pixel=False, stack_size=1,
               sensors={"rgb_camera": (RGBCamera, 640, 360)})
    cfg["vehicle_config"]["image_source"] = "rgb_camera"
env = MetaDriveEnv(cfg)

CLASSES = {"clear": "A", "overtake": "B", "blocked": "A", "return": "C", "keep_left": "B"}


def scene(kind):
    cam = a.view == "camera"
    if kind == "clear":
        return L_RIGHT, []
    if kind == "overtake":
        return L_RIGHT, [(rng.uniform(20, 60), L_RIGHT)]
    if kind == "blocked":
        left_dx = rng.uniform(12, 35) if cam else rng.uniform(-6, 8)
        return L_RIGHT, [(rng.uniform(20, 60), L_RIGHT), (left_dx, L_LEFT)]
    if kind == "return":
        return L_LEFT, [(rng.uniform(-35, -12), L_RIGHT)]
    if kind == "keep_left":
        return L_LEFT, [(rng.uniform(8, 25) if cam else rng.uniform(3, 20), L_RIGHT)]
    raise KeyError(kind)


labels = []
for kind, label in CLASSES.items():
    for i in range(a.n):
        env.reset()
        ego_y, obs = scene(kind)
        env.agent.set_position((EGO_X, ego_y))
        env.agent.set_heading_theta(0.0)
        spawned = [env.engine.spawn_object(DefaultVehicle, vehicle_config={}, position=(EGO_X + dx, ly), heading=0.0)
                   for dx, ly in obs]
        for _ in range(2):
            o, *_ = env.step([0.0, 0.0])
        if a.view == "camera":
            img = np.asarray(o["image"][..., -1]).astype(np.uint8)
            if a.flip:
                img = img[..., ::-1]
        else:
            W, H, S = 600, 1200, 6
            full = np.asarray(env.render(mode="topdown", window=False, screen_size=(W, H), scaling=S,
                                         target_agent_heading_up=True)).astype(np.uint8)
            cx, cy = W // 2, H // 2
            img = full[cy - 90 * S: cy + 40 * S, cx - 7 * S: cx + 13 * S]
        name = f"{kind}_{i}.png"
        Image.fromarray(np.ascontiguousarray(img)).save(os.path.join(a.out, name))
        labels.append(dict(
            file=name, kind=kind, label=label,
            ego_lane="RIGHT" if ego_y == L_RIGHT else "LEFT",
            current_skill="CRUISE_RIGHT" if ego_y == L_RIGHT else "OVERTAKE_LEFT",
            right_ahead=any(ly == L_RIGHT and dx > 0 for dx, ly in obs),
            left_car=any(ly == L_LEFT for dx, ly in obs),
            obstacles=[dict(dx=round(float(dx), 1), lane="RIGHT" if ly == L_RIGHT else "LEFT") for dx, ly in obs]))
        if spawned:
            env.engine.clear_objects([v.id for v in spawned])
    print(f"{a.view}: {kind} x{a.n}", flush=True)

json.dump(labels, open(os.path.join(a.out, "labels.json"), "w"), indent=1)
env.close()

In [ ]:
%%writefile /content/vlm_exp.py
"""Round-2 VLM router experiments. One model per process (GPU memory is freed on exit).

usage: python vlm_exp.py --model Qwen/Qwen2.5-VL-3B-Instruct --frames /content/frames2 --out results_3b.json

For each view (topdown, camera) found under --frames/<view>/{test,shots}:
  direct      zero-shot: image -> one letter A/B/C
  fewshot     5 labelled example images (1 per class) in the fixed prefix -> one letter. The prefix is cached.
  perceive    two yes/no questions about the scene (Q1 car ahead in RIGHT lane? Q2 other car in LEFT lane?)
              -> a fixed rule maps the answers to a skill. Q2 reuses Q1's cached image.
"""
import argparse
import base64
import io
import json
import math
import os
import random
import statistics as st
import time

from PIL import Image

ap = argparse.ArgumentParser()
ap.add_argument("--model", required=True)
ap.add_argument("--frames", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--max-model-len", type=int, default=8192)
args = ap.parse_args()

# Do not touch CUDA before LLM(): if CUDA is already initialised, vLLM starts its engine with "spawn", which
# re-imports this script and crashes. Read the GPU's compute capability from nvidia-smi instead.
import subprocess  # noqa: E402
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "fork")
import torch  # noqa: E402
from vllm import LLM, SamplingParams  # noqa: E402

random.seed(0)
CC = float(subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0])
DTYPE = "bfloat16" if CC >= 8 else "half"
kw = dict(model=args.model, dtype=DTYPE, max_model_len=args.max_model_len, limit_mm_per_prompt={"image": 6},
          enable_prefix_caching=True, gpu_memory_utilization=0.88, seed=0)
if "Qwen2.5-VL" in args.model:
    kw["mm_processor_kwargs"] = {"min_pixels": 28 * 28, "max_pixels": 1600 * 28 * 28}
t0 = time.time()
llm = LLM(**kw)
LOAD_S = time.time() - t0
tok = llm.get_tokenizer()


def first_id(s):
    return tok.encode(s, add_special_tokens=False)[0]


LETTERS = ["A", "B", "C"]
SP_LETTER = SamplingParams(temperature=0, max_tokens=1, allowed_token_ids=[first_id(c) for c in LETTERS], logprobs=5)
SP_YESNO = SamplingParams(temperature=0, max_tokens=1, allowed_token_ids=[first_id("Yes"), first_id("No")], logprobs=5)

SCENE = {
    "topdown": ("You see a bird's-eye view of the road. It shows our side of a highway: the yellow line on the left "
                "is the road centre, then the LEFT lane, a dashed white divider, then the RIGHT lane, then the road "
                "edge. Our car is GREEN, near the bottom, driving UP the image. Every other car is parked (speed 0)."),
    "camera": ("You see the view from a forward-facing camera on our car. Our side of the road has two lanes: a LEFT "
               "lane and a RIGHT lane. Every other car you see is parked (speed 0)."),
}
SKILLS = ("Choose the skill our car should run now:\n"
          "A = CRUISE_RIGHT: stay in / go to the right lane and keep driving. Use this when the right lane ahead is "
          "clear, OR when a car blocks the right lane ahead but the left lane is occupied (we must wait).\n"
          "B = OVERTAKE_LEFT: move to / stay in the left lane. Use this when a car blocks the right lane ahead (or is "
          "still ahead of us on the right) and the left lane is free.\n"
          "C = RETURN_RIGHT: go back to the right lane. Use this when we are in the left lane and the right lane "
          "ahead is clear.\n\nReply with exactly one letter: A, B or C.")
PERCEIVE = "Answer the question about the image with exactly one word: Yes or No."
Q1 = "Question: Is there a car ahead of us in the RIGHT lane?"
Q2 = "Question: Is there another car (not ours) in the LEFT lane?"


def system(view, task):
    return f"You are the perception and planning module of a self-driving car. {SCENE[view]}\n\n" + (
        SKILLS if task == "skill" else PERCEIVE)


def uri(img):
    b = io.BytesIO()
    img.save(b, format="PNG")
    return "data:image/png;base64," + base64.b64encode(b.getvalue()).decode()


def user(img, text):
    return {"role": "user", "content": [{"type": "image_url", "image_url": {"url": uri(img)}}, {"type": "text", "text": text}]}


def state(lab):
    return f"Our car is in the {lab['ego_lane']} lane. Current skill: {lab['current_skill']}. Ego speed: 20 m/s."


def run(msgs, sp):
    t = time.perf_counter()
    out = llm.chat(msgs, sp, use_tqdm=False)[0]
    ms = (time.perf_counter() - t) * 1000
    o = out.outputs[0]
    conf = None
    if o.logprobs:
        lp = {v.decoded_token.strip(): v.logprob for v in o.logprobs[0].values()}
        z = {k: math.exp(v) for k, v in lp.items()}
        conf = round(z.get(o.text.strip(), 0.0) / (sum(z.values()) or 1.0), 3)
    return o.text.strip(), ms, len(out.prompt_token_ids), conf


def fresh(img):
    """Change one pixel so every test frame misses the image cache, like a new camera frame."""
    img = img.copy()
    img.putpixel((random.randrange(img.width), random.randrange(img.height)), (random.randrange(256),) * 3)
    return img


def rule(ego_lane, right_ahead, left_car):
    if ego_lane == "RIGHT":
        return "B" if (right_ahead and not left_car) else "A"
    return "B" if right_ahead else "C"


results = dict(model=args.model, gpu=torch.cuda.get_device_name(0), load_s=round(LOAD_S, 1), views={})
for view in ("topdown", "camera"):
    vdir = os.path.join(args.frames, view)
    if not os.path.exists(os.path.join(vdir, "test", "labels.json")):
        print(f"[{view}] no frames, skipping", flush=True)
        continue
    test = json.load(open(os.path.join(vdir, "test", "labels.json")))
    shots = json.load(open(os.path.join(vdir, "shots", "labels.json")))
    load = lambda d, f: Image.open(os.path.join(vdir, d, f)).convert("RGB")
    sys_skill, sys_perc = system(view, "skill"), system(view, "perceive")
    # Few-shot prefix: one labelled example per class, identical on every call -> served from the prefix cache.
    seen, fs_prefix = set(), [{"role": "system", "content": sys_skill}]
    for s in shots:
        if s["kind"] in seen:
            continue
        seen.add(s["kind"])
        fs_prefix += [user(load("shots", s["file"]), state(s)), {"role": "assistant", "content": s["label"]}]

    # Warm-up (kernels, CUDA graphs, and caching the fixed prefixes).
    w = load("test", test[0]["file"])
    for _ in range(2):
        run([{"role": "system", "content": sys_skill}, user(fresh(w), state(test[0]))], SP_LETTER)
        run(fs_prefix + [user(fresh(w), state(test[0]))], SP_LETTER)
        run([{"role": "system", "content": sys_perc}, user(fresh(w), Q1)], SP_YESNO)

    rows = []
    for lab in test:
        img = fresh(load("test", lab["file"]))
        r = dict(file=lab["file"], kind=lab["kind"], want=lab["label"], right_ahead=lab["right_ahead"], left_car=lab["left_car"])
        r["direct"], r["direct_ms"], r["direct_tok"], r["direct_conf"] = run(
            [{"role": "system", "content": sys_skill}, user(img, state(lab))], SP_LETTER)
        # Each method gets its own pixel-perturbed copy, so no call reuses another's vision-encoder output.
        r["fewshot"], r["fewshot_ms"], r["fewshot_tok"], r["fewshot_conf"] = run(
            fs_prefix + [user(fresh(img), state(lab))], SP_LETTER)
        img2 = fresh(img)  # new pixels again so Q1 pays for the image; Q2 then reuses Q1's cached image
        a1, r["q1_ms"], r["q1_tok"], r["q1_conf"] = run([{"role": "system", "content": sys_perc}, user(img2, Q1)], SP_YESNO)
        a2, r["q2_ms"], _, r["q2_conf"] = run([{"role": "system", "content": sys_perc}, user(img2, Q2)], SP_YESNO)
        r["q1"], r["q2"] = a1.lower().startswith("y"), a2.lower().startswith("y")
        r["perceive"] = rule(lab["ego_lane"], r["q1"], r["q2"])
        for m in ("direct", "fewshot", "perceive"):
            r[f"{m}_ok"] = r[m][:1] == lab["label"]
        r["q1_ok"], r["q2_ok"] = r["q1"] == lab["right_ahead"], r["q2"] == lab["left_car"]
        rows.append(r)
        print(f"[{view}] {lab['file']:15s} want {lab['label']}  direct {r['direct']} fewshot {r['fewshot']} "
              f"perceive {r['perceive']} (Q1 {int(r['q1'])}/{int(lab['right_ahead'])} Q2 {int(r['q2'])}/{int(lab['left_car'])})  "
              f"ms {r['direct_ms']:.0f}/{r['fewshot_ms']:.0f}/{r['q1_ms']:.0f}+{r['q2_ms']:.0f}", flush=True)

    acc = lambda k: round(sum(r[k] for r in rows) / len(rows), 3)
    med = lambda k: round(st.median(r[k] for r in rows), 1)
    summ = dict(
        n=len(rows),
        direct=dict(acc=acc("direct_ok"), ms=med("direct_ms"), prompt_tok=med("direct_tok")),
        fewshot=dict(acc=acc("fewshot_ok"), ms=med("fewshot_ms"), prompt_tok=med("fewshot_tok")),
        perceive=dict(acc=acc("perceive_ok"), ms=round(st.median(r["q1_ms"] + r["q2_ms"] for r in rows), 1),
                      q1_acc=acc("q1_ok"), q2_acc=acc("q2_ok"), q1_ms=med("q1_ms"), q2_ms_cached_image=med("q2_ms")),
    )
    results["views"][view] = dict(summary=summ, rows=rows)
    print(f"[{view}] SUMMARY {json.dumps(summ)}", flush=True)

json.dump(results, open(args.out, "w"), indent=1)
print("saved", args.out)

## 2. MetaDrive in its own Python 3.11 environment

Kept separate from vLLM so their package versions never clash.

In [ ]:
!uv venv -q -p 3.11 /content/md
!uv pip install -q --python /content/md/bin/python metadrive-simulator opencv-python-headless
!/content/md/bin/python -m metadrive.pull_asset 2>&1 | tail -1

## 3. Render frames

Test set: 10 per class × 5 classes = 50 frames per view (seed 1). Few-shot examples: 1 per class (seed 99, so never in the test set).
The camera tries headless GPU rendering first, then falls back to a virtual display (`xvfb-run`).

In [ ]:
import subprocess, os
PY = "/content/md/bin/python"
# Colab ships the NVIDIA driver but not the file that tells libEGL where it is -> headless GPU rendering fails.
os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
with open("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "w") as f:
    f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
ENV = dict(os.environ, LD_LIBRARY_PATH="/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", ""))
XVFB = ["xvfb-run", "-a", "-s", "-screen 0 1280x720x24"]
TRIES = [("default", {}, []), ("EGL/GPU", {"P3D_DISPLAY": "p3headlessgl"}, []),
         ("xvfb/CPU", {"P3D_DISPLAY": "pandagl"}, XVFB)]

def gen(view, split, n, seed):
    out = f"/content/frames2/{view}/{split}"
    cmd = [PY, "/content/gen_frames2.py", "--view", view, "--out", out, "--n", str(n), "--seed", str(seed)]
    for name, extra, prefix in TRIES:
        p = subprocess.run(prefix + cmd, capture_output=True, text=True, env={**ENV, **extra})
        if p.returncode == 0 and os.path.exists(f"{out}/labels.json"):
            print(f"{view}/{split}: ok ({name})")
            return True
        print(f"{view}/{split}: failed ({name}):", (p.stderr or p.stdout)[-300:])
    return False

for view in ("topdown", "camera"):
    ok = gen(view, "test", 10, 1) and gen(view, "shots", 1, 99)
    if not ok:
        print(f"*** {view} frames unavailable; that view will be skipped ***")

### Look at the frames before trusting any numbers

Check: our car / road visible, parked cars where the class says they are. If the **camera sky looks orange**
(colours swapped), rerun step 3 with `--flip 0` added to `cmd`.

In [ ]:
import json, matplotlib.pyplot as plt
from PIL import Image
for view in ("topdown", "camera"):
    d = f"/content/frames2/{view}/test"
    if not os.path.exists(f"{d}/labels.json"):
        continue
    labs = json.load(open(f"{d}/labels.json"))
    picks = [next(l for l in labs if l["kind"] == k) for k in ("clear", "overtake", "blocked", "return", "keep_left")]
    fig, ax = plt.subplots(1, 5, figsize=(18, 4 if view == "camera" else 8))
    for a, l in zip(ax, picks):
        a.imshow(Image.open(f"{d}/{l['file']}")); a.axis("off")
        a.set_title(f"{l['kind']} → {l['label']}\nego {l['ego_lane']}, {[(o['lane'], o['dx']) for o in l['obstacles']]}", fontsize=8)
    fig.suptitle(view); plt.show()

## 4. Run the models

Each model runs in its own process, so GPU memory is freed between models. The log prints every frame:
`want` is the right answer, `Q1 a/b` means the model said a, the truth is b (1 = yes), and `ms` is direct/fewshot/Q1+Q2.

In [ ]:
!cd /content && python vlm_exp.py --model Qwen/Qwen2.5-VL-3B-Instruct --frames /content/frames2 --out res_qwen25vl_3b.json 2>&1 | tee log_3b.txt | grep -E '^\[|saved|Error|error' 

In [ ]:
!cd /content && python vlm_exp.py --model Qwen/Qwen2.5-VL-7B-Instruct --frames /content/frames2 --out res_qwen25vl_7b.json 2>&1 | tee log_7b.txt | grep -E '^\[|saved|Error|error' 

Optional: a newer model family. Needs a recent vLLM. If it errors, skip it; the summary works without it.

In [ ]:
!cd /content && python vlm_exp.py --model Qwen/Qwen3-VL-8B-Instruct --frames /content/frames2 --out res_qwen3vl_8b.json 2>&1 | tee log_q3_8b.txt | grep -E '^\[|saved|Error|error' 

## 5. Summary

In [ ]:
import glob, pandas as pd
recs, per_class = [], []
for f in sorted(glob.glob("/content/res_*.json")):
    d = json.load(open(f))
    name = d["model"].split("/")[-1]
    for view, v in d["views"].items():
        s = v["summary"]
        for m in ("direct", "fewshot", "perceive"):
            recs.append(dict(model=name, view=view, method=m, accuracy=s[m]["acc"], median_ms=s[m]["ms"],
                             prompt_tok=s[m].get("prompt_tok"),
                             q1_acc=s[m].get("q1_acc"), q2_acc=s[m].get("q2_acc")))
        for r in v["rows"]:
            for m in ("direct", "fewshot", "perceive"):
                per_class.append(dict(model=name, view=view, method=m, kind=r["kind"], ok=r[f"{m}_ok"]))
summary = pd.DataFrame(recs)
display(summary)
display(pd.DataFrame(per_class).pivot_table(index=["model", "view", "method"], columns="kind", values="ok", aggfunc="mean").round(2))
print("Chance reference: always-A scores 0.40.  Perception: Q1 = car ahead in RIGHT lane, Q2 = other car in LEFT lane.")

## 6. Download the results (one zip, send it back)

In [ ]:
from google.colab import files
summary.to_csv("/content/summary.csv", index=False)
!cd /content && rm -f round2_results.zip && zip -qr round2_results.zip res_*.json log_*.txt summary.csv frames2/*/test/labels.json frames2/camera/test/clear_0.png frames2/camera/test/overtake_0.png frames2/camera/test/blocked_0.png frames2/camera/test/return_0.png frames2/camera/test/keep_left_0.png
files.download("/content/round2_results.zip")